# Day 3 · Pandas 数据读取、筛选与清洗

目标：读懂真实数据，编写有依据的清洗规则，完成数据质量报告。
学习顺序：基础 60 分钟 → 深度 40 分钟 → 广度 20 分钟 → 作业 30 分钟；Bonus 另加 30 分钟。
沿用前两天 Markdown + Code + Assignment 格式。每段先猜结果再运行，参考答案在末尾。

### 1. list / ndarray / Series / DataFrame

DataFrame 是有行列标签的二维表，单列通常是 Series。
本周数据来自 [UCI Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance)。
使用数学课程文件，395 行、33 列；CSV 用分号分隔，原始数据没有缺失值。
G1/G2/G3 是三个阶段的成绩，均为 0～20；本练习用 10 分为及格线。
studytime 是 1～4 的档位，不能直接当作小时。
字段定义见 ../data/raw/student.txt。

先固定保留验证/测试数据，今天只探索训练集。拆分的理由在第五天详细讲解。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course/bootcamp/week1/data/raw/student-mat.csv").exists())
raw = pd.read_csv(ROOT / "course/bootcamp/week1/data/raw/student-mat.csv", sep=";")
# 固定划分，第三至七天始终使用相同学生；实际行数为 276/59/60。
train, heldout = train_test_split(raw, test_size=0.30, random_state=42)
val, test = train_test_split(heldout, test_size=0.50, random_state=42)
train, val, test = train.copy(), val.copy(), test.copy()
print("训练/验证/测试：", len(train), len(val), len(test))


In [ ]:
df = train.copy()
display(df.head())
print(df.shape)
df.info()
display(df.describe())
display(df.select_dtypes(exclude="number").describe())
display(df.isna().sum())
print("重复行数：", df.duplicated().sum())

### 2. 选择、筛选、排序

df["G3"] 是 Series，df[["G3"]] 是 DataFrame。
loc 按行列标签选择；iloc 按位置选择。训练集保留原始行号，所以位置 0 不一定对应标签 0。
切片 iloc 的结束位置不包含在内；loc 标签切片在存在对应端点时包含结束标签。

In [ ]:
print(type(df["G3"]), type(df[["G3"]]))
display(df.iloc[:3, :5])
display(df.loc[:, ["school", "studytime", "G3"]].head())
display(df.loc[(df["G3"] >= 10) & (df["studytime"] >= 3),
               ["school", "studytime", "G3"]].head())
display(df.sort_values("G3", ascending=False).head())
print("第一行标签：", df.index[0])

Assignment 1：筛选 G2 ≥ 12 且 G3 < 10 的记录，仅显示 G1/G2/G3。解释单层中括号和双层中括号对形状的影响。

In [ ]:
# 在这里完成 Assignment 1。


### 3. 制造教学脏数据，记录清洗前状态

只在训练集前 12 行的副本上注入错误，再复制一行：
缺失 age/G2、非法 studytime、99 分、school 的空格及大小写、缺失标签。
这是人为构造的教学数据，不是原始数据质量结论。

In [ ]:
# 错误仅注入训练集的副本，不改动原始文件。
dirty = train.head(12)[["school", "age", "studytime", "G1", "G2", "G3"]].copy()
dirty.insert(0, "record_id", dirty.index)
dirty = dirty.reset_index(drop=True)
dirty.loc[0, "age"] = np.nan
dirty.loc[1, "G2"] = np.nan
dirty["studytime"] = dirty["studytime"].astype("str")
dirty.loc[2, "studytime"] = "unknown"
dirty.loc[3, "G3"] = 99
dirty.loc[4, "school"] = " gp "
dirty.loc[5, "G3"] = np.nan
dirty = pd.concat([dirty, dirty.iloc[[6]]], ignore_index=True)
display(dirty)


In [ ]:
quality_before = pd.DataFrame({"dtype": dirty.dtypes.astype(str),
                               "missing": dirty.isna().sum(),
                               "unique": dirty.nunique()})
display(quality_before)
print("重复 ID：", dirty["record_id"].duplicated().sum())
print("无效或缺失标签：", (~dirty["G3"].between(0, 20)).sum())

### 4. 深度：清洗要有依据

- 字符串先 strip 去空格，再 upper 统一大小写。
- to_numeric(errors="coerce") 将不能解析的数字变成缺失，便于统一处理。
- 超范围目标记为缺失；缺失标签行删除，不能编造真实成绩。
- 仅删除明确的人为重复记录，不把“特征相同”自动等同于“同一学生”。
- 特征缺失用训练集的中位数填补；验证/测试集不能贡献填充值。
- G3=0 合法，保留。箱线图上的离群点也不能自动删除。

正式训练会用 Pipeline 让填补与模型绑定。下列函数不会改变输入对象。

In [ ]:
def clean_students(frame, medians):
    result = frame.copy()
    result["school"] = result["school"].str.strip().str.upper()
    for col in ["age", "studytime", "G1", "G2", "G3"]:
        result[col] = pd.to_numeric(result[col], errors="coerce")
    result.loc[~result["G3"].between(0, 20), "G3"] = np.nan
    # 此例的重复 ID 已知来自复制；真实数据必须先确认记录身份。
    result = result.drop_duplicates("record_id").dropna(subset=["G3"])
    result[medians.index] = result[medians.index].fillna(medians)
    return result.reset_index(drop=True)
medians = train[["age", "studytime", "G1", "G2"]].median()
clean = clean_students(dirty, medians)
assert len(clean) == 10
assert clean["record_id"].is_unique
assert clean["G3"].between(0, 20).all()
assert not clean.isna().any().any()
display(clean)


In [ ]:
display(pd.Series({"before_rows": len(dirty), "after_rows": len(clean),
                   "removed": len(dirty)-len(clean),
                   "remaining_missing": int(clean.isna().sum().sum())}))

### 5. 广度：分组统计与连接

groupby 先分组再聚合。比较均值时同时报告人数。
merge 依赖键匹配；validate="many_to_one" 要求右表键唯一，防止误复制行。

In [ ]:
display(df.groupby("studytime")["G3"].agg(["count", "mean", "median", "std"]))
lookup = pd.DataFrame({"school": ["GP", "MS"], "label": ["School GP", "School MS"]})
joined = df.merge(lookup, on="school", how="left", validate="many_to_one")
assert len(joined) == len(df)
display(joined[["school", "label", "G3"]].head())
output = ROOT / "course/bootcamp/week1/outputs"
output.mkdir(exist_ok=True)
clean.to_csv(output / "day3_cleaned_practice.csv", index=False)
reread = pd.read_csv(output / "day3_cleaned_practice.csv")
assert len(reread) == len(clean)
display(reread.head())

### 6. 深度：索引对齐与安全修改

Series 运算会按标签对齐，不是简单按位置配对。修改表时使用 loc，避免链式赋值。

In [ ]:
a = pd.Series([10, 20], index=["A", "B"])
b = pd.Series([1, 2], index=["B", "C"])
display(a + b)
practice = df[["G3"]].copy()
practice["passed"] = practice["G3"] >= 10
practice.loc[practice["G3"] < 10, "passed"] = False
display(practice.head())

### 7. DAY 3 ASSIGNMENT

1. 基础：完成 Assignment 1，按 school 输出人数、均分、及格率。
2. 深度：用 Markdown 写出每条清洗规则及理由，说明缺失、零和离群值的区别。
3. 广度：读取导出文件，检查行数、字段与类型；为何 CSV 不保留所有 dtype 信息？
4. Bonus：验证清洗幂等性，即 clean(clean(x)) 与 clean(x) 相同；验证输入不变。
5. 验收：练习副本 13 行变为 10 行；ID 唯一、无缺失、目标范围合法。
6. 交付：本 Notebook 中的质量报告、分组表和至少三条文字观察。

后续建模仍使用原始数据，不能拿这个只有 10 行的副本代替真实训练集。

In [ ]:
# 作业区


### 学习总结（填写）

- 清洗前后有哪些变化：
- 删除记录可能带来什么偏差：
- 哪些统计量只能从训练集学习：
- 今天仍然不理解的地方：

### 参考答案 · 先完成作业再看

In [ ]:
display(df.loc[(df["G2"] >= 12) & (df["G3"] < 10), ["G1", "G2", "G3"]])
display(df.assign(passed=df["G3"] >= 10).groupby("school").agg(
    count=("G3", "size"), mean_score=("G3", "mean"), pass_rate=("passed", "mean")))
before = dirty.copy(deep=True)
once = clean_students(dirty, medians)
pd.testing.assert_frame_equal(before, dirty)
pd.testing.assert_frame_equal(once, clean_students(once, medians))
print("输入不变、清洗幂等性验证通过。")